# Overlay detector — full pipeline on Colab GPU

Two-stream (Bayar/SRM forensic noise + RGB backbone) face-region fraud detector.
Runs the full pipeline: **data download -> face-crop caching -> train -> infer -> submit**.

**Colab Pro** recommended: A100/V100 GPU, longer runtime, more RAM.
Also works on free-tier T4 (reduce `batch_size` to 16 if you hit OOM).

---

### Two ways to use this notebook

**Option A — Run directly in Colab:** Upload this `.ipynb` to Colab, set runtime to GPU, run all cells top-to-bottom.

**Option B — IDE remote kernel:** Paste the five `[Colab N]` cells below into a fresh Colab notebook
(Runtime > GPU), run them to start the tunnel, then connect this notebook's kernel to the printed URL
and skip to **STEP 2**.

---

## Option B — Colab browser cells (paste into Colab, run once)

New Colab notebook -> Runtime type **GPU (T4/A100)** -> add Secrets `NGROK_AUTH_TOKEN`, `KAGGLE_TOKEN`
(token from kaggle.com/settings/api, starts with `KGAT_`).
Paste each cell below in order.

```python
# [Colab 1] GPU check + clone feat/overlay-detector + install
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU -> set Runtime to GPU')
!pip install -q pyngrok kagglehub
!git clone --depth 1 -b feat/overlay-detector https://github.com/hyunsoochung-portfolio/kaggle_freuid_2026.git
%cd kaggle_freuid_2026
!pip install -q -e .   # reuses Colab's built-in torch (CUDA); installs timm, facenet-pytorch, etc.
```

```python
# [Colab 2] Kaggle auth via access token (Colab Secret: KAGGLE_TOKEN)
import os
from google.colab import userdata
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(userdata.get('KAGGLE_TOKEN').strip())
os.chmod('/root/.kaggle/access_token', 0o600)
print('access_token written')
```

```python
# [Colab 3] Download competition data via kagglehub -> data/raw/
import kagglehub, os
DATA_DIR = 'data/raw'
if not os.path.exists(f'{DATA_DIR}/train_labels.csv'):
    path = kagglehub.competition_download('the-freuid-challenge-2026-ijcai-ecai')
    print('downloaded to', path)
    os.makedirs(DATA_DIR, exist_ok=True)
    !cp -r {path}/* {DATA_DIR}/
    print('copied to', DATA_DIR)
else:
    print('data already present, skipping download')
!echo 'train imgs:' && ls data/raw/train/train | wc -l
```

```python
# [Colab 4] Mount Google Drive -> checkpoints/ + submissions/ auto-save to Drive
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/freuid/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/freuid/submissions', exist_ok=True)
!rm -rf checkpoints submissions
!ln -s /content/drive/MyDrive/freuid/checkpoints checkpoints
!ln -s /content/drive/MyDrive/freuid/submissions submissions
print('checkpoints/, submissions/ -> linked to Drive')
```

```python
# [Colab 5] Start Jupyter server + ngrok tunnel -> print URL
import subprocess, time, socket
from google.colab import userdata
from pyngrok import conf, ngrok
conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN').strip()
JUPYTER_TOKEN = 'freuid'
for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
ngrok.kill()
log = open('/tmp/jl.log', 'w')
subprocess.Popen(['jupyter','server','--ip=0.0.0.0','--port=8888','--no-browser','--allow-root',
    f'--ServerApp.token={JUPYTER_TOKEN}','--ServerApp.password=','--ServerApp.allow_origin=*',
    '--ServerApp.disable_check_xsrf=True','--ServerApp.allow_remote_access=True',
    '--ServerApp.root_dir=/content/kaggle_freuid_2026'], stdout=log, stderr=subprocess.STDOUT)
for i in range(30):
    time.sleep(1)
    s = socket.socket()
    if s.connect_ex(('127.0.0.1', 8888)) == 0: s.close(); break
    s.close()
print(ngrok.connect(8888, 'http').public_url + '/?token=' + JUPYTER_TOKEN)
```

Copy the printed URL. In your IDE: top-right kernel picker -> `Select Another Kernel...` -> `Existing Jupyter Server...` -> paste URL -> `Python 3`.
Keep the Colab tab open -- closing it ends the session.

**Then skip to STEP 2** (repo is already cloned and data is downloaded).

---

## STEP 1 — Environment setup (Option A only)

Skip this section if you used the Option B Colab browser cells above.

In [ ]:
# [1] GPU check + clone feat/overlay-detector + install
import torch, os

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
else:
    raise RuntimeError('No GPU -- set Runtime > Change runtime type > GPU (A100 for Colab Pro)')

if not os.path.exists('/content/kaggle_freuid_2026'):
    !git clone --depth 1 -b feat/overlay-detector https://github.com/hyunsoochung-portfolio/kaggle_freuid_2026.git /content/kaggle_freuid_2026

%cd /content/kaggle_freuid_2026
!pip install -q kagglehub
!pip install -q -e .

In [ ]:
# [2] Kaggle auth via access token (Colab Secret: KAGGLE_TOKEN)
import os
from google.colab import userdata

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(userdata.get('KAGGLE_TOKEN').strip())
os.chmod('/root/.kaggle/access_token', 0o600)
print('access_token written')

In [ ]:
# [3] Download competition data via kagglehub -> data/raw/
import kagglehub, os

DATA_DIR = 'data/raw'
if not os.path.exists(f'{DATA_DIR}/train_labels.csv'):
    path = kagglehub.competition_download('the-freuid-challenge-2026-ijcai-ecai')
    print('downloaded to', path)
    os.makedirs(DATA_DIR, exist_ok=True)
    !cp -r {path}/* {DATA_DIR}/
    print('copied to', DATA_DIR)
else:
    print('data already present, skipping download')

!echo 'train images:' && ls data/raw/train/train | wc -l
!echo 'test images:'  && ls data/raw/public_test/public_test | wc -l

In [ ]:
# [4] Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/freuid/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/freuid/submissions', exist_ok=True)

!rm -rf checkpoints submissions
!ln -s /content/drive/MyDrive/freuid/checkpoints checkpoints
!ln -s /content/drive/MyDrive/freuid/submissions submissions
print('checkpoints/, submissions/ -> linked to Drive')

## STEP 2 — Connection check + configure

Option B users start here. Verify the Colab environment looks right before writing the config.

In [ ]:
# Connection check: GPU + Drive link + data present?
import torch, os
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu  (<- runtime is not GPU!)')
print('cwd:', os.getcwd())
print('checkpoints -> Drive?', os.path.realpath('checkpoints'))
!echo 'train images:' && ls data/raw/train/train | wc -l

device: cuda
cwd: /content/kaggle_freuid_2026
checkpoints -> Drive? /content/kaggle_freuid_2026/checkpoints
train images:
ls: cannot access 'data/raw': No such file or directory
0


Adjust `batch_size` for your GPU:

| GPU | VRAM | `batch_size` |
|-----|------|--------------|
| A100 | 40 GB | 64 |
| V100 | 16 GB | 32 |
| T4 | 16 GB | 16 |

In [ ]:
%%writefile configs/overlay_colab.yaml
# Overlay detector -- Colab config (adjust batch_size per GPU table above)
name: overlay_colab
seed: 42

data_dir: data/raw
image_size: 224
val_fraction: 0.1
epochs: 5
batch_size: 32
lr: 0.0001
weight_decay: 0.0001
num_workers: 0   # MTCNN singleton is not fork-safe

model_type: overlay

model:
  type: overlay
  noise_frontend: bayar
  noise_feat_dim: 128
  use_rgb_stream: true
  rgb_backbone: resnet34
  rgb_pretrained: true
  fusion_dim: 128

overlay:
  crop_margin: 0.75
  crop_cache_dir: /content/overlay_crops  # local SSD — fast random access during training
  # detect_long_side: 1024  # downscale long side to this before MTCNN detection; never upscales
  # min_face_size: 60       # MTCNN min_face_size — higher prunes more pyramid levels (faster on large IDs)

## STEP 3 — Prepare face crops on local SSD

Crops live on Google Drive for persistence but must be on local SSD for fast training.
This cell handles both cases automatically:
- **First session**: runs MTCNN on the train set (~15 min on A100), saves crops to local SSD, then copies to Drive for future sessions.
- **Subsequent sessions**: copies crops from Drive to local SSD (~2–3 min), then training reads at full SSD speed.

Test crops are precached automatically at the start of inference (STEP 5).

In [ ]:
from freuid.config import load_config
from freuid.data import precache_crops

cfg = load_config('configs/overlay_colab.yaml')
precache_crops(cfg, splits=("train",))  # val is a subset of train files — no separate pass needed

## STEP 4 — Train

In [ ]:
!python -m freuid.train --config configs/overlay_colab.yaml

## STEP 5 — Inference

In [ ]:
CKPT = 'checkpoints/overlay_colab.pt'
OUT  = 'submissions/overlay_colab.csv'

# backbone / image_size come from the checkpoint -- no --config needed
!python -m freuid.infer --checkpoint {CKPT} --out {OUT}
!head -3 {OUT} && echo '---' && wc -l {OUT}

## STEP 6 — Submit to Kaggle

In [ ]:
OUT = 'submissions/overlay_colab.csv'
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m 'overlay detector (colab)'